# Baseline Residual Analysis

**Objective**: Analyze the performance of the Gated Hybrid Baseline model using Out-of-Fold (OOF) predictions. We focus on identifying where the Trend model succeeds and where the Residual model (LGBM) and the Gate logic (`is_closed`) struggle.

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns

# Set Plotly rendering to notebook
import plotly.io as pio
pio.renderers.default = "notebook_connected"

# Constants
OOF_PATH = Path("../../artifacts/baseline/oof_predictions.csv")
RAW_PATH = Path("../../data/raw")

# Load OOF Data
df = pd.read_csv(OOF_PATH, parse_dates=["date"])

# Load Metadata for enrichment (Holidays)
holidays = pd.read_csv(RAW_PATH / "holidays_events.csv", parse_dates=["date"])
# Simplify holidays: flag any record as a holiday day
holiday_days = holidays[holidays['transferred'] == False]['date'].unique()
df['is_holiday'] = df['date'].isin(holiday_days).astype(str) # String for better plotting labels

print(f"Loaded {len(df)} OOF predictions enriched with holiday metadata.")
df.head()

## 1. Interpretations of Model Architecture

### Figure 1: Trend vs Residual (Log Space)
- **Ideal Alignment**: The horizontal OLS trendline at $Y=0$ indicates that the Residual model is **unbiased**.
- **Feature Separation**: The Linear model handles the big time-based trends, while the LGBM handles the zero-centered deviations.

### Figure 2: Distribution of Log Predictions
- **Trend (Blue)**: Captures the scale and tiers of sales volumes.
- **Residual (Red)**: Tightly centered on zero, acting as a "fine-tuner."

In [ ]:
# Trend vs Residual Correlation Visualization
fig = px.scatter(df.sample(10000), x="trend_log", y="residual_log", 
                 title="Validation: Trend vs Residual Unbiasedness", 
                 opacity=0.1, trendline="ols", trendline_color_override="red")
fig.show()

## 2. Global Performance & Time series Error

Calculating global metrics and reviewing the overall error distribution.

In [ ]:
global_rmsle = np.sqrt(df['sq_log_error'].mean())
print(f"Global OOF RMSLE: {global_rmsle:.4f}")

# RMSLE over Time
error_time = df.groupby('date')['sq_log_error'].mean().apply(np.sqrt).reset_index()
fig = px.line(error_time, x='date', y='sq_log_error', title="Global RMSLE Evolution (OOF Window)")
fig.show()

## 3. Residual Distribution Analysis (Boxplots)

Instead of just means, we look at the spread of errors across categories.

In [ ]:
def plot_residual_dist(df, group_col, title):
    # We use log1p(residual_abs) to visualize variance without extreme outliers crushing the scale
    fig = px.box(df.sample(20000), x=group_col, y="residual", 
                 title=f"Residual Distribution (Pred-Actual) by {title}",
                 points=False) # Hide points for speed
    fig.add_hline(y=0, line_dash="dash", line_color="red")
    return fig

# 1. Holiday vs Non-Holiday
plot_residual_dist(df, 'is_holiday', 'Holiday Presence').show()

# 2. Store Type
plot_residual_dist(df, 'type', 'Store Type').show()

# 3. Cluster
plot_residual_dist(df, 'cluster', 'Cluster').show()

## 4. Residual Dynamics Over Time

Analyzing how error patterns evolve for different segments.

In [ ]:
def plot_temporal_rmsle(df, group_col, title):
    agg = df.groupby(['date', group_col])['sq_log_error'].mean().apply(np.sqrt).reset_index()
    fig = px.line(agg, x='date', y='sq_log_error', color=group_col, 
                  title=f"Temporal RMSLE by {title}")
    return fig

# 1. By Holiday
plot_temporal_rmsle(df, 'is_holiday', 'Holiday').show()

# 2. By City (Top 5 for readability)
top_cities = df.groupby('city')['sq_log_error'].mean().sort_values(ascending=False).head(5).index
plot_temporal_rmsle(df[df['city'].isin(top_cities)], 'city', 'Top 5 High-Error Cities').show()

# 3. By Cluster
plot_temporal_rmsle(df, 'cluster', 'Cluster').show()

## 5. Store 38 Deep Dive

Visualization of Actual vs Predicted sales for Store 38.

In [ ]:
store_38 = df[df['store_nbr'] == 38].groupby('date')[['sales', 'sales_pred']].sum().reset_index()

fig = go.Figure()
fig.add_trace(go.Scatter(x=store_38['date'], y=store_38['sales'], name='Actual Sales', line=dict(color='blue')))
fig.add_trace(go.Scatter(x=store_38['date'], y=store_38['sales_pred'], name='Predicted Sales', line=dict(color='orange', dash='dash')))

fig.update_layout(
    title="Store 38: Aggregated Actual vs Predicted Sales",
    xaxis_title="Date",
    yaxis_title="Total Sales",
    hovermode="x unified"
)
fig.show()

## 6. Gate Analysis (`is_closed`)

Reviewing cases where the model predicted zero sales but actual sales were positive.

In [ ]:
false_closed = df[(df['sales_pred'] == 0) & (df['sales'] > 0)]
print(f"Number of False Closed cases (Pred=0, Actual>0): {len(false_closed)}")
print(f"Total Lost Sales due to False Closed: {false_closed['sales'].sum():.2f}")

if len(false_closed) > 0:
    worst_false_closed_stores = false_closed.groupby('store_nbr')['sales'].sum().sort_values(ascending=False).head(10)
    print("Top Stores with False Closures (Sales Lost):")
    print(worst_false_closed_stores)